# 02 — Synthetic anomaly validation

**Production Pulse — Phase 3.** Injects each synthetic anomaly family into held-out fault-free TEP data and **measures** whether it is *individually in range but jointly abnormal*.

**Hypothesis under test (not assumed):** for a family to be *SUPPORTED*, on held-out data
1. the injection creates no new out-of-range values (≥ 99 % of anomalous rows),
2. the single-feature rule check does no better than its own false-alarm rate (recall − FPR ≤ 0.05),
3. a multivariate detector catches it (recall ≥ 0.5).

The multivariate detector here is the **PCA reconstruction-error baseline**; the autoencoder comes in Phase 4. `max_marginal_auc` / `marginal_auc_excess` are an extra, stricter check: can *any single feature on its own* separate the modified rows from normal rows? Because TEP signals are strongly autocorrelated, the AUC is also computed on the **same rows before injection**; `marginally_hidden` uses the excess (injected − clean ≤ 0.05), so chance differences between time windows are not mistaken for an injection effect.

Data roles (all run-disjoint): **fit** = training-file train split · **calibrate** = training-file validation split · **inject & evaluate** = a seeded subset of the **evaluation-role** runs of the separate fault-free **testing** file (the testing file is split into calibration 0.2 / evaluation 0.7 / holdout 0.1; calibration and holdout runs are never selected here).

> **Correction (2026-10-21).** An earlier version of this notebook drew its 40 runs from *all 500* testing runs, ignoring the role split. With seed 42 that selection included 4 **holdout** runs (42, 196, 241, 304) and 10 calibration runs. Those runs were only scored, never used to fit, train or calibrate anything, but the 4 holdout runs are no longer unseen and are excluded from demo selections (`HOLDOUT_RUNS_PREVIOUSLY_SCORED`). Results printed by the earlier version, including `reports/executed/02_synthetic_anomaly_validation.executed.ipynb`, come from the old selection and are kept as historical evidence only. This corrected notebook draws a different set of runs, so it has to be re-run to produce results for the corrected selection; none exist yet.

⚠️ This notebook has not been run yet; no results exist until you run it with the `.venv313` kernel. Large faulty files are never loaded.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents)
                     if (p / "pyproject.toml").is_file() and (p / "src" / "production_pulse").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the project folder.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd

from production_pulse import get_paths, load_config, setup_logging
from production_pulse.data import FEATURE_COLUMNS, METADATA_COLUMNS, load_raw_dataset, split_metadata_features
from production_pulse.preprocessing import FeatureScaler
from production_pulse.splitting import HOLDOUT_RUNS_PREVIOUSLY_SCORED, apply_split, select_runs, split_runs
from production_pulse.anomaly_generation import FAMILIES, AnomalyGenerator, AnomalySpec
from production_pulse.baselines import PCAReconstructionDetector, SingleFeatureThresholdDetector
from production_pulse.evaluation import HypothesisCriteria, assess_family_detectability, compare_detectors

paths = get_paths()
config = load_config(root=paths.root)
logger = setup_logging(config=config, paths=paths)

# ---- experiment settings (change here, then Run All) ----
SEED = config.random_seed          # 42 by default
N_EVAL_RUNS = 40                   # runs drawn from the EVALUATION role of the fault-free TESTING file
ANOMALY_RATE = 0.05                # fraction of evaluation rows made anomalous, per family
TARGET_FPR = 0.01                  # PCA threshold: false-alarm rate on calibration-normal data
PCA_VARIANCE = 0.90                # fraction of training variance kept by PCA
CRITERIA = HypothesisCriteria()    # explicit pass criteria (see evaluation.py)
SAVE_OUTPUTS = False               # True -> tables to reports/, figures to reports/figures/
F = list(FEATURE_COLUMNS)
print("Project root:", paths.root, "| seed:", SEED)

## 1. Data: run-level split, training-only scaling, held-out evaluation runs

In [ ]:
train_raw = load_raw_dataset("fault_free_training", paths=paths, config=config)
split = split_runs(train_raw, config.split.fractions(), seed=SEED, group_columns=config.split.group_columns)
parts = apply_split(train_raw, split)

scaler = FeatureScaler().fit(parts["train"], source="fault_free_training / train split")
train_x = scaler.transform_frame(parts["train"])          # fit data (generator, rule limits, PCA)
val_x = scaler.transform_frame(parts["validation"])       # calibration-normal data (PCA threshold)

test_raw = load_raw_dataset("fault_free_testing", paths=paths, config=config)
# Same seeded run-role split as workflow.prepare_fault_free_data (calibration / evaluation / holdout).
test_split = split_runs(test_raw, config.evaluation.fractions(), seed=SEED, group_columns=config.split.group_columns)
# Runs come ONLY from the evaluation role. Calibration and holdout runs can never be drawn.
eval_runs = select_runs(test_split, "evaluation", N_EVAL_RUNS, seed=SEED, exclude=HOLDOUT_RUNS_PREVIOUSLY_SCORED)
assert set(eval_runs) <= set(test_split.runs("evaluation"))
assert not set(eval_runs) & (set(test_split.runs("calibration")) | set(test_split.runs("holdout")))
eval_raw = (test_raw[test_raw["simulationRun"].isin(eval_runs)]
            .sort_values(["simulationRun", "sample"]).reset_index(drop=True))
eval_meta, _ = split_metadata_features(eval_raw)
eval_x = scaler.transform_frame(eval_raw)                 # clean held-out data (scaled with TRAIN statistics)

print(f"fit rows {len(train_x):,} | calibration rows {len(val_x):,} | evaluation rows {len(eval_x):,} "
      f"({N_EVAL_RUNS} of {len(test_split.runs('evaluation'))} evaluation-role runs of the testing file)")
print("first evaluation runs:", eval_runs[:10])
del test_raw

## 2. Baselines — fitted on training-normal only, PCA threshold calibrated on validation-normal

In [ ]:
single = SingleFeatureThresholdDetector().fit(train_x)
pca = PCAReconstructionDetector(n_components=PCA_VARIANCE).fit(train_x)
pca.calibrate(val_x, target_fpr=TARGET_FPR)
print("PCA components kept:", pca.n_components_,
      f"({pca.explained_variance_ratio_.sum():.3f} of training variance)")
for det in (single, pca):
    print(det.threshold_info())

clean_single, clean_pca = single.detect(eval_x), pca.detect(eval_x)
print(f"False-alarm rate on CLEAN held-out data: rule check {clean_single.predictions.mean():.4f}, "
      f"PCA {clean_pca.predictions.mean():.4f}")

## 3. Anomaly generator — fitted on training-normal only

In [ ]:
gen = AnomalyGenerator().fit(train_x)
print(f"{gen.anchor_candidates_.size} features have a partner with |r| >= {gen.correlation_threshold}")
gen.correlated_pairs(top=10).round(3)

## 4. Experiment — one family at a time on the same held-out runs
Each family is injected separately into a fresh copy of the same clean evaluation data with the same seed, so all detectors see the same examples and labels.

In [ ]:
results, assessments, comparisons = {}, [], []
for family in FAMILIES:
    r = gen.generate(eval_x, AnomalySpec(family, ANOMALY_RATE), seed=SEED, groups=eval_meta["simulationRun"])
    d_single, d_pca = single.detect(r.features), pca.detect(r.features)
    results[family] = (r, d_single, d_pca)
    a = assess_family_detectability(r.row_info, d_single, d_pca, features=r.features, events=r.events,
                                    clean_features=eval_x, criteria=CRITERIA)
    modified_values = (r.events["length"] * r.events["n_modified_features"]).sum()
    a["clipped_value_fraction"] = r.events["n_new_range_violations"].sum() / modified_values
    assessments.append(a)
    c = compare_detectors(r.labels, {"rule_check": d_single, "pca": d_pca})
    c.insert(0, "family", family)
    comparisons.append(c)

assessment = pd.concat(assessments)
comparison = pd.concat(comparisons)
print("Verdicts below are computed from the measured numbers in the same table.")
assessment[["n_anomalous_rows", "in_range_fraction", "clipped_value_fraction", "single_feature_recall",
            "single_feature_fpr", "multivariate_recall", "multivariate_fpr", "max_marginal_auc",
            "marginal_auc_excess", "most_separating_feature", "marginally_hidden", "verdict", "reasons"]].round(4)

In [ ]:
# Same held-out examples & labels for both detectors, per family
comparison.reset_index(names="detector").set_index(["family", "detector"])[
    ["precision", "recall", "f1", "false_positive_rate", "pr_auc", "roc_auc", "tp", "fp", "fn"]].round(4)

In [ ]:
failed = assessment[assessment["verdict"] != "SUPPORTED"]
print("Families NOT supported:" if len(failed) else "All families met the criteria on this sample.")
for fam, row in failed.iterrows():
    print(f"  - {fam}: {row['reasons']}")
hidden = assessment["marginally_hidden"]
print("\nSupported AND no single feature separates them (strict):",
      list(assessment.index[(assessment["verdict"] == "SUPPORTED") & (hidden == True)]))

## 5. Severity sweep — how strong must an anomaly be before PCA notices it?

In [ ]:
sweep = []
for family, magnitudes in {"minor_component_shift": [3, 6, 10, 15, 25], "correlation_break": [0.25, 0.5, 1.0],
                           "coordinated_shift": [0.5, 1.0, 1.5, 2.5]}.items():
    for m in magnitudes:
        r = gen.generate(eval_x, AnomalySpec(family, ANOMALY_RATE, magnitude=m), seed=SEED, groups=eval_meta["simulationRun"])
        a = assess_family_detectability(r.row_info, single.detect(r.features), pca.detect(r.features), criteria=CRITERIA)
        row = a.iloc[0]
        sweep.append({"family": family, "magnitude": m, "rule_recall": row["single_feature_recall"],
                      "pca_recall": row["multivariate_recall"], "verdict": row["verdict"]})
pd.DataFrame(sweep).round(4)

## 6. Plots — feature-wise distributions vs joint patterns
Needs matplotlib (installed in `.venv313` 3.11.2 per your report). Change `PLOT_FAMILY` to inspect another family.

In [ ]:
import matplotlib.pyplot as plt

PLOT_FAMILY = "correlation_break"

def save(fig, name):
    if SAVE_OUTPUTS:
        from production_pulse.paths import safe_output_path
        fig.savefig(safe_output_path(paths.figures_dir / name, paths, overwrite=True), dpi=150, bbox_inches="tight")

r, d_single, d_pca = results[PLOT_FAMILY]
ev = r.events.iloc[0]
modified = ev["modified_features"].split(";")
anchor = ev["anchor_feature"] or modified[0]
partner = modified[0] if modified[0] != anchor else modified[1]
fam_rows = (r.row_info["family"] == PLOT_FAMILY).to_numpy()
normal_rows = (r.row_info["is_anomaly"] == 0).to_numpy()
rng_tab = gen.ranges()
print("event 0:", ev[["group", "length", "anchor_feature", "modified_features"]].to_dict())

In [ ]:
# 6a. Feature-wise view: each feature's anomalous values vs its normal distribution and range
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, feat in zip(axes, [anchor, partner]):
    ax.hist(r.features.loc[normal_rows, feat], bins=80, density=True, alpha=0.5, label="normal")
    ax.hist(r.features.loc[fam_rows, feat], bins=80, density=True, alpha=0.6, label=f"{PLOT_FAMILY}")
    for v in rng_tab.loc[feat]:
        ax.axvline(v, color="k", ls="--", lw=0.8)
    ax.set_title(f"{feat} (dashed = normal range)")
axes[0].legend()
fig.suptitle("Single features: anomalous values sit inside the normal range")
fig.tight_layout(); save(fig, f"phase3_{PLOT_FAMILY}_marginals.png"); plt.show()

In [ ]:
# 6b. Joint view: the same two features plotted against each other
sample = np.random.default_rng(SEED).choice(np.flatnonzero(normal_rows), 5000, replace=False)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(r.features[anchor].to_numpy()[sample], r.features[partner].to_numpy()[sample], s=3, alpha=0.3, label="normal")
ax.scatter(r.features.loc[fam_rows, anchor], r.features.loc[fam_rows, partner], s=6, alpha=0.7, label=PLOT_FAMILY)
ax.set_xlabel(anchor); ax.set_ylabel(partner); ax.legend()
ax.set_title(f"Joint pattern (training r = {gen.corr_[F.index(anchor), F.index(partner)]:.2f})")
fig.tight_layout(); save(fig, f"phase3_{PLOT_FAMILY}_joint.png"); plt.show()

In [ ]:
# 6c. Detector scores per family: rule-check score (>1 = outside a limit) and PCA reconstruction error
fig, axes = plt.subplots(2, len(FAMILIES), figsize=(3.2 * len(FAMILIES), 5.5), sharey="row")
for j, (family, (r_f, ds, dp)) in enumerate(results.items()):
    lab = r_f.labels.to_numpy() == 1
    for i, (det, name) in enumerate([(ds, "rule score"), (dp, "PCA SPE")]):
        ax = axes[i, j]
        vals = np.log10(det.scores + 1e-12)
        ax.hist(vals[~lab], bins=60, density=True, alpha=0.5, label="normal")
        ax.hist(vals[lab], bins=60, density=True, alpha=0.6, label="anomalous")
        ax.axvline(np.log10(det.threshold), color="k", ls="--", lw=0.8)
        ax.set_title(f"{family}\n{name}" if i == 0 else name, fontsize=8)
axes[0, 0].legend(fontsize=7)
fig.supxlabel("log10(score); dashed = threshold")
fig.tight_layout(); save(fig, "phase3_detector_scores.png"); plt.show()

In [ ]:
if SAVE_OUTPUTS:
    from production_pulse.paths import safe_output_path
    assessment.to_csv(safe_output_path(paths.reports_dir / "phase3_family_assessment.csv", paths, overwrite=True))
    comparison.to_csv(safe_output_path(paths.reports_dir / "phase3_detector_comparison.csv", paths, overwrite=True))
    pd.DataFrame(sweep).to_csv(safe_output_path(paths.reports_dir / "phase3_severity_sweep.csv", paths, overwrite=True))
    print("saved tables to", paths.reports_dir)
else:
    print("SAVE_OUTPUTS is False - nothing written.")

## 7. Record in the research logbook
- Copy the assessment table: which families were **SUPPORTED / NOT SUPPORTED**, and why.
- Which families are in range but still separable by a single feature (`marginally_hidden = False`)?
- The severity at which PCA starts catching each family (section 5).
- Caveats: PCA is a stand-in for the multivariate detector until the Phase 4 autoencoder; results are for one seed and 40 evaluation-role runs. If you compare with the earlier executed copy, remember it used a different (uncorrected) run selection.